# FreqAI data exploration and feature engineering

This notebook demonstrates how to explore data, engineer features, and train prediction models using the Freqtrade FreqAI framework.
It follows a typical workflow to prototype strategies and models before deploying them in backtesting or live trading.

## 1. Setup working directory

In [ ]:
import os
from pathlib import Path

# Modify this path to point to your local freqtrade repository
project_root = Path.cwd()
assert (project_root / 'LICENSE').is_file(), 'Run this notebook from the repository root.'
datadir = project_root / 'user_data' / 'data' / 'binance'
datadir

## 2. Load strategy and configuration

In [ ]:
from freqtrade.templates.FreqaiExampleStrategy import FreqaiExampleStrategy
from freqtrade.freqai.data_kitchen import DataKitchen
from freqtrade.freqai.freqai_interface import FreqaiInterface

# Minimal configuration dictionary for FreqAI
config = {
    'timeframe': '5m',
    'stake_currency': 'USDT',
    'datadir': str(datadir),
    'freqai': {
        'enabled': True,
        'label_period': 12,
        'feature_parameters': {
            'include_timeframes': ['5m'],
            'include_corr_pairs': [],
            'include_shifted_candles': 0,
            'indicator_periods_candles': [14]
        },
        'model_type': 'LightGBMClassifier'
    }
}

strategy = FreqaiExampleStrategy(config)

## 3. Load data and populate indicators

In [ ]:
from freqtrade.data.history import load_pair_history
from pandas import DataFrame

pair = 'BTC/USDT'
start = '2022-01-01'
end = '2022-06-01'

data = load_pair_history(datadir=datadir, timeframe=config['timeframe'], pair=pair, start_date=start, end_date=end)
df = strategy.advise_indicators(data, {'pair': pair, 'tf': config['timeframe']})
df = strategy.set_freqai_targets(df, {'pair': pair, 'tf': config['timeframe']})
df.dropna(inplace=True)
df.head()

## 4. Visualize data

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(15,5))
df['close'].plot(ax=ax, label='Close')
df['%-rsi-14'].plot(ax=ax, label='RSI')
ax.set_title(f'{pair} price and RSI')
ax.legend()
plt.show()

## 5. Prepare features and labels

In [ ]:
feature_columns = [c for c in df.columns if c.startswith('%')]
label_column = [c for c in df.columns if c.startswith('&')][0]
X = df[feature_columns]
y = df[label_column]
X.shape, y.shape

## 6. Train / test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, shuffle=False)
X_train.shape, X_test.shape

## 7. Train LightGBM model

In [ ]:
import lightgbm as lgb

model = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], eval_metric='logloss', verbose=20)
y_pred = model.predict_proba(X_test)[:,1]

## 8. Evaluate performance

In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, y_pred)
print(f'AUC: {auc:.3f}')
lgb.plot_importance(model, max_num_features=20)
plt.show()

## 9. Plot predictions vs. actual

In [ ]:
fig, ax = plt.subplots(figsize=(15,5))
ax.plot(y_test.index, y_test.values, label='Actual')
ax.plot(y_test.index, y_pred, label='Predicted')
ax.set_title('Model prediction vs. target')
ax.legend()
plt.show()

## 10. Next steps

- Experiment with custom features in the strategy's `feature_engineering_*` functions.
- Modify the model and its hyperparameters within the config.
- Add cross-validation, walk-forward analysis, or hyperopt for further optimization.